# Experimento 3: herramienta con contratos

**Herramienta.** `plazos.computar_vencimiento` calcula la fecha de vencimiento de un plazo con las reglas del artículo 183 del Código Civil y la excepción del artículo 2007 para la caducidad. Se eligió porque las contradicciones de la PC1 eran sobre todo de plazos (10 días hábiles frente a 30 días calendario), y comparar dos plazos exige primero computarlos.

**Pregunta.** ¿La cadena de contratos clasifica cada fallo en el tipo correcto y reintenta solo cuando un reintento puede cambiar el resultado?

Supuesto declarado: un día es hábil si es de lunes a viernes y no es feriado nacional. El Código Civil no define día hábil. El calendario cargado cubre los 16 feriados nacionales de 2026; un vencimiento fuera de 2026 es un error de dominio, no un cálculo silencioso.

La herramienta es independiente del RAG. No hay agente ni LLM que la invoque.

## 0. Entorno

In [1]:
!python -m pip install -q --break-system-packages -r ../requirements.txt

In [2]:
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

import config, corpus, metricas
print("raíz:", ROOT)
print("RUN_REAL_RETRIEVAL:", config.RUN_REAL_RETRIEVAL, "| RUN_REAL_RERANKER:", config.RUN_REAL_RERANKER,
      "| RUN_REAL_LLM:", config.RUN_REAL_LLM)

raíz: /workspace/CC-0F4/Evaluaciones/parcial
RUN_REAL_RETRIEVAL: True | RUN_REAL_RERANKER: True | RUN_REAL_LLM: True


In [3]:
import asyncio
import herramienta_plazos as hp

print(json.dumps(hp.TOOL_SPEC, ensure_ascii=False, indent=2))

{
  "type": "function",
  "function": {
    "name": "plazos.computar_vencimiento",
    "description": "Calcula la fecha de vencimiento de un plazo según el art. 183 del Código Civil (y el art. 2007 para caducidad), con el calendario de feriados nacionales 2026.",
    "parameters": {
      "additionalProperties": false,
      "properties": {
        "fecha_inicio": {
          "description": "Día en que empieza el plazo (por ejemplo, la notificación).",
          "format": "date",
          "title": "Fecha Inicio",
          "type": "string"
        },
        "cantidad": {
          "description": "Número de unidades del plazo.",
          "maximum": 3650,
          "minimum": 1,
          "title": "Cantidad",
          "type": "integer"
        },
        "unidad": {
          "enum": [
            "dias",
            "meses",
            "anios"
          ],
          "title": "Unidad",
          "type": "string"
        },
        "computo_dias": {
          "default": "naturales",


## 1. Contratos

```text
contrato de entrada  -> tipos, rangos, enums, sin campos extra (Pydantic)
validación de dominio -> combinaciones válidas en tipos pero sin sentido jurídico, cobertura del calendario
contrato de salida   -> la herramienta solo responde si su salida cumple PlazoOutput
```

Política de ejecución: hasta 3 intentos, espera de 50 y 100 ms entre intentos, 200 ms por intento y 800 ms de límite total.

| Tipo de error | ¿Se reintenta? | Razón |
|---|---|---|
| protocol | no | la herramienta no está permitida; repetir no la vuelve permitida |
| validation | no | los mismos argumentos vuelven a fallar el contrato |
| domain | no | la regla jurídica o el calendario no cambian entre intentos |
| transient | sí | la fuente del calendario puede responder en el siguiente intento |
| timeout | sí | un intento lento puede terminar a tiempo en otro, si queda límite total |
| output_contract | no | una salida inválida es un defecto de la implementación |
| execution | no | error no previsto; reintentar ocultaría el defecto |

## 2. Ejecución de los 13 casos

In [4]:
cases = corpus.read_jsonl(config.DATA_DIR / "casos_herramienta.jsonl")
runs = []
for case in cases:
    run = await hp.ejecutar(case["tool"], case["arguments"], case["fault"])
    res, exp = run["result"], case["expected"]
    ok = res["status"] == exp["status"] and run["attempts"] == exp["attempts"]
    if exp["status"] == "ok":
        ok = ok and res["data"]["fecha_vencimiento"] == exp["fecha_vencimiento"]
    else:
        ok = ok and res["error"]["kind"] == exp["kind"] and res["error"]["code"] == exp.get("code", res["error"]["code"])
    runs.append({"case_id": case["case_id"], "tipo": case["tipo"], **run, "esperado": exp, "correcto": ok})

with open("results/exp3_ejecuciones.jsonl", "w", encoding="utf-8") as fh:
    for r in runs:
        fh.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")

table = pd.DataFrame([{
    "case_id": r["case_id"], "tipo": r["tipo"], "fallo_inyectado": r["fault"], "estado": r["result"]["status"],
    "tipo_error": (r["result"]["error"] or {}).get("kind"), "codigo": (r["result"]["error"] or {}).get("code"),
    "reintentable": (r["result"]["error"] or {}).get("retryable"),
    "vencimiento": (r["result"]["data"] or {}).get("fecha_vencimiento"),
    "intentos": r["attempts"], "latencia_ms": r["latency_ms"], "correcto": r["correcto"]} for r in runs])
table.to_csv("results/exp3_resumen.csv", index=False)
print("casos con el resultado esperado:", int(table.correcto.sum()), "de", len(table))
table

casos con el resultado esperado: 13 de 13


,case_id,tipo,fallo_inyectado,estado,tipo_error,codigo,reintentable,vencimiento,intentos,latencia_ms,correcto
0,H01,ejecucion_valida,ninguna,ok,None,None,None,2026-11-02,1,8.6,True
1,H02,ejecucion_valida,ninguna,ok,None,None,None,2026-10-16,1,7.7,True
2,H03,ejecucion_valida,ninguna,ok,None,None,None,2026-10-31,1,7.7,True
3,H04,ejecucion_valida,ninguna,ok,None,None,None,2026-03-02,1,8.3,True
4,H05,entrada_invalida,ninguna,error,validation,entrada_invalida,False,None,0,0.1,True
5,H06,entrada_invalida,ninguna,error,validation,entrada_invalida,False,None,0,0.1,True
6,H07,error_dominio,ninguna,error,domain,habiles_requiere_dias,False,None,1,0.1,True
7,H08,error_dominio,ninguna,error,domain,fuera_de_calendario,False,None,1,7.7,True
8,H09,fallo_transitorio,transitorio_una_vez,ok,None,None,None,2026-11-02,2,59.9,True
9,H10,fallo_transitorio,transitorio_siempre,error,transient,calendario_no_disponible,True,None,3,151.2,True


## 3. Trazas por intento

In [5]:
traces = pd.DataFrame([{"case_id": r["case_id"], **t} for r in runs for t in r["traces"]])
traces.to_csv("results/exp3_trazas.csv", index=False)
traces

,case_id,attempt,outcome,ms
0,H01,1,ok,8.3
1,H02,1,ok,7.4
2,H03,1,ok,7.6
3,H04,1,ok,8.1
4,H07,1,domain,0.1
5,H08,1,domain,7.7
6,H09,1,transient,0.1
7,H09,2,ok,9.1
8,H10,1,transient,0.1
9,H10,2,transient,0.1


## 4. Detalle de una ejecución válida y de un error de dominio

In [6]:
for cid in ["H02", "H08"]:
    r = next(x for x in runs if x["case_id"] == cid)
    print(cid, json.dumps(r["args"], ensure_ascii=False))
    print(json.dumps(r["result"], ensure_ascii=False, indent=2), "\n")

H02 {"fecha_inicio": "2026-10-01", "cantidad": 10, "unidad": "dias", "computo_dias": "habiles"}
{
  "status": "ok",
  "data": {
    "fecha_vencimiento": "2026-10-16",
    "fecha_vencimiento_nominal": "2026-10-16",
    "dia_semana_vencimiento": "viernes",
    "dias_inhabiles_saltados": 5,
    "reglas_aplicadas": [
      "art. 183 inc. 1: plazo por días hábiles, establecido por la ley o el acto",
      "art. 183 inc. 4: excluye el día inicial e incluye el día del vencimiento"
    ],
    "fundamento": [
      "cc_art_183"
    ]
  },
  "error": null
} 

H08 {"fecha_inicio": "2026-12-20", "cantidad": 30, "unidad": "dias"}
{
  "status": "error",
  "data": null,
  "error": {
    "code": "fuera_de_calendario",
    "kind": "domain",
    "retryable": false,
    "message": "El vencimiento sale del calendario 2026."
  }
} 



## 5. Pruebas automáticas

Las mismas comprobaciones están en `tests/test_herramienta_plazos.py` y corren con `pytest`.

In [7]:
!python -m pytest -q --color=no -p no:cacheprovider tests

....................                                                     [100%]
20 passed in 4.01s
